# Сеть плана квартиры: обучение на Colab

Сеть по скану квартиры (E57) размечает план: **улица, помещение, стена, дверь, окно, проём, мусор, пустота**
(проём в перекрытии и лестница). Кроме классов она предсказывает расстояние до грани стены, направление
стены и высоты проёмов. Вход - 8 каналов с пикселем 2 см (`simscan/netinput.py`), сеть - U-Net на ResNet-34
(`simscan/net.py`).

**Перед запуском**
1. *Среда выполнения → Сменить тип среды → GPU* (T4 хватает; L4/A100 быстрее).
2. На Google Drive в папке `plannet` (путь меняется в настройках ниже):
   * `dataset.zip` - датасет: `python -m simscan make-dataset ... --out dataset` и
     `python -m simscan dataset pack dataset --out dataset.zip` на машине с ядрами;
   * `real/*.e57` - реальные объекты для проверки (не обязательно).
3. *Среда выполнения → Выполнить все.*

Прогон пишется на Drive в `plannet/runs/<имя>`: `last.pt` - после каждой эпохи, `best.pt` - лучшая по
val, `history.jsonl` - кривые. **Если сессия оборвалась** - снова *Выполнить все*: обучение продолжится
с `last.pt`. Чтобы начать заново - другое имя прогона.

In [ ]:
#@title Настройки { display-mode: "form" }
DRIVE_DIR = '/content/drive/MyDrive/plannet'  #@param {type:"string"}
DATASET_ZIP = 'dataset.zip'  #@param {type:"string"}
RUN_NAME = 'r1'  #@param {type:"string"}
EPOCHS = 60  #@param {type:"integer"}
BATCH = 8  #@param {type:"integer"}
CROP = 512  #@param {type:"integer"}
LR = 0.001  #@param {type:"number"}
WORKERS = 2  #@param {type:"integer"}
BRANCH = 'claude/serene-einstein-9avsce'  #@param {type:"string"}

import os, sys
IN_COLAB = 'google.colab' in sys.modules
# проверка вне Colab: PLANNET_LOCAL=<папка, как plannet на Drive>, PLANNET_REPO=<код>, PLANNET_WORK=<рабочая папка>
if not IN_COLAB and os.environ.get('PLANNET_LOCAL'):
    DRIVE_DIR = os.environ['PLANNET_LOCAL']
WORK = os.environ.get('PLANNET_WORK', '/content')
REPO = os.environ.get('PLANNET_REPO', f'{WORK}/DotCloudCONVERT')
DATA = f'{WORK}/dataset'
RUN = f'{DRIVE_DIR}/runs/{RUN_NAME}'
for k in ('EPOCHS', 'BATCH', 'CROP', 'WORKERS'):          # для быстрой проверки ноутбука
    if os.environ.get('PLANNET_' + k):
        globals()[k] = int(os.environ['PLANNET_' + k])
print('Drive:', DRIVE_DIR, '| прогон:', RUN)

In [ ]:
#@title Google Drive и GPU
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
import torch
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU: {p.name}, {p.total_memory / 2**30:.0f} ГБ')
else:
    print('GPU нет - обучение на CPU будет очень медленным. Среда выполнения → Сменить тип среды → GPU.')
os.makedirs(RUN, exist_ok=True)
assert os.path.exists(f'{DRIVE_DIR}/{DATASET_ZIP}') or os.path.exists(f'{DATA}/meta.json'), \
    f'нет {DRIVE_DIR}/{DATASET_ZIP}'

In [ ]:
#@title Код (ветка репозитория)
import subprocess

def run(cmd, cwd=None):
    """Команда с выводом по строкам (видно ход обучения); ошибка - исключение."""
    p = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    if p.wait():
        raise RuntimeError(f'код {p.returncode}: {" ".join(cmd)}')

if not os.path.exists(f'{REPO}/simscan'):
    run(['git', 'clone', '-q', '-b', BRANCH, 'https://github.com/BykovVadimCV/DotCloudCONVERT', REPO])
elif IN_COLAB:
    run(['git', '-C', REPO, 'pull', '-q'])
sys.path.insert(0, REPO)
os.chdir(REPO)
import cv2, numpy as np                                    # есть в Colab; обучению больше ничего не нужно
from simscan.netinput import CLASSES, CHANNELS
NAMES = ['улица', 'помещение', 'стена', 'дверь', 'окно', 'проём', 'мусор', 'пустота']
run(['git', '-C', REPO, 'log', '-1', '--format=код: %h %s'])

In [ ]:
#@title Датасет: с Drive на локальный диск Colab
import json, shutil, time, zipfile
if not os.path.exists(f'{DATA}/meta.json'):
    t = time.time()
    local_zip = f'{WORK}/{DATASET_ZIP}'
    if not os.path.exists(local_zip):
        print('копирую с Drive...'); shutil.copy(f'{DRIVE_DIR}/{DATASET_ZIP}', local_zip)
    with zipfile.ZipFile(local_zip) as z:
        top = z.namelist()[0].split('/')[0]
        print('распаковываю', len(z.namelist()), 'файлов...'); z.extractall(WORK)
    if os.path.abspath(f'{WORK}/{top}') != os.path.abspath(DATA):
        os.rename(f'{WORK}/{top}', DATA)
    os.remove(local_zip)
    print(f'готово за {time.time() - t:.0f} с')
meta = json.load(open(f'{DATA}/meta.json'))
splits = {s: open(f'{DATA}/splits/{s}.txt').read().split() for s in ('train', 'val', 'test')}
print('уровней:', {k: len(v) for k, v in splits.items()}, '| пиксель', meta['pixel_m'], 'м')
assert splits['train'] and splits['val'], 'в датасете нет train или val'
# состав: квартиры под отделку, двухуровневые, доля классов на выборке
scn = [json.load(open(f'{DATA}/scenes/{i}/scene.json')) for i in splits['train'][:400]]
print(f"под отделку {np.mean([bool(s.get('bare')) for s in scn]):.0%}, "
      f"уровней двухуровневых квартир {np.mean([s.get('levels_in_scene', 1) > 1 for s in scn]):.0%}")
cnt = np.zeros(256)
for i in splits['train'][:200]:
    cnt += np.bincount(cv2.imread(f'{DATA}/scenes/{i}/sem.png', cv2.IMREAD_UNCHANGED).ravel(), minlength=256)
print('доля пикселей:', {n: f'{c / cnt[:8].sum():.1%}' for n, c in zip(NAMES, cnt[:8])})

In [ ]:
#@title Как выглядят образцы (вход и разметка)
import matplotlib.pyplot as plt
from simscan.infer import COLORS
from simscan.torchdata import FloorplanSamples

def rgb(x):
    """8 каналов -> картинка: R - занятость внизу, G - пол виден, B - занятость вверху."""
    x = np.asarray(x, np.float32)
    return np.clip(np.stack([x[0], x[3] * 0.6 + x[6] * 0.4, x[2]], -1), 0, 1)

def sem_rgb(s):
    s = np.asarray(s).copy(); ign = s == 255; s[ign] = 0
    im = COLORS[s].copy(); im[ign] = (255, 255, 255); return im

ds = FloorplanSamples(DATA, 'train', crop=CROP, train=True, seed=1)
fig, ax = plt.subplots(2, 4, figsize=(18, 9))
for k, idx in enumerate(np.random.default_rng(0).permutation(len(ds))[:4]):
    s = ds[int(idx)]
    ax[0, k].imshow(rgb(s['input'])); ax[0, k].set_title(s['id'], fontsize=8)
    ax[1, k].imshow(sem_rgb(s['sem']))
for a in ax.ravel(): a.axis('off')
from matplotlib.patches import Patch
fig.legend([Patch(color=COLORS[i] / 255) for i in range(8)], NAMES, loc='lower center', ncol=8)
plt.show()

In [ ]:
#@title Обучение (продолжает с last.pt, если он есть)
cmd = [sys.executable, '-m', 'simscan', 'train', '--data', DATA, '--out', RUN, '--epochs', str(EPOCHS),
       '--batch', str(BATCH), '--crop', str(CROP), '--lr', str(LR), '--workers', str(WORKERS), '--resume']
if os.environ.get('PLANNET_MAX_STEPS'):
    cmd += ['--max-steps', os.environ['PLANNET_MAX_STEPS']]
run(cmd)

In [ ]:
#@title Кривые обучения
h = [json.loads(l) for l in open(f'{RUN}/history.jsonl')]
fig, ax = plt.subplots(1, 3, figsize=(18, 4))
ep = [r['epoch'] for r in h]
ax[0].plot(ep, [r['total'] for r in h], label='train'); ax[0].plot(ep, [r.get('val_total') for r in h], label='val')
ax[0].set_title('потеря'); ax[0].legend()
ax[1].plot(ep, [r.get('miou_walls') for r in h]); ax[1].set_title('val mIoU: стена, дверь, окно, проём')
iou = np.array([[np.nan if v is None else v for v in r.get('iou', [None] * 8)] for r in h], float)
for c in range(8):
    ax[2].plot(ep, iou[:, c], label=NAMES[c])
ax[2].set_title('val IoU по классам'); ax[2].legend(fontsize=7)
plt.show()
best = max(h, key=lambda r: r.get('miou_walls', -1))
print(f"лучшая эпоха {best['epoch']}: mIoU стен {best.get('miou_walls', 0):.3f}")

In [ ]:
#@title Проверка best.pt на val: картинки и IoU по классам на уровнях целиком
from simscan.infer import load_model, predict
from simscan.torchdata import load_level
model, dev = load_model(f'{RUN}/best.pt')
cm = np.zeros((8, 8), np.int64)
show = splits['val'][:4]
fig, ax = plt.subplots(len(show), 3, figsize=(15, 5 * len(show)), squeeze=False)
for k, i in enumerate(splits['val']):
    s = load_level(f'{DATA}/scenes/{i}')
    pred = predict(model, dev, s['input'], tile=CROP)['prob'].argmax(0)
    m = s['sem'] != 255
    cm += np.bincount(s['sem'][m] * 8 + pred[m], minlength=64).reshape(8, 8)
    if i in show:
        ax[k, 0].imshow(rgb(s['input'])); ax[k, 0].set_title(i, fontsize=8)
        ax[k, 1].imshow(sem_rgb(s['sem'])); ax[k, 1].set_title('разметка')
        ax[k, 2].imshow(sem_rgb(pred)); ax[k, 2].set_title('сеть')
for a in ax.ravel(): a.axis('off')
plt.show()
tp = np.diag(cm); iou = tp / np.maximum(cm.sum(0) + cm.sum(1) - tp, 1)
for n, v, c in zip(NAMES, iou, cm.sum(1)):
    print(f'{n:10s} IoU {v:.3f}' + ('' if c else '  (нет в val)'))

In [ ]:
#@title Реальные объекты: E57 из plannet/real на Drive
import glob
from IPython.display import Image, display
real = sorted(glob.glob(f'{DRIVE_DIR}/real/*.e57'))
if not real:
    print(f'нет файлов {DRIVE_DIR}/real/*.e57 - положите реальные сканы, чтобы посмотреть сеть на них')
else:
    run([sys.executable, '-m', 'simscan', 'infer', f'{RUN}/best.pt', *real, '--out', f'{RUN}/pred', '--tile', str(CROP)])
    for p in sorted(glob.glob(f'{RUN}/pred/*/L*/pred.png')):
        print(p); display(Image(p, width=900))
    print('карты классов, вероятности и привязка к координатам - там же, в', f'{RUN}/pred')

**Где результат.** Веса - `plannet/runs/<имя>/best.pt` на Drive; применять так:
`python -m simscan infer best.pt объект.e57 --out pred` (нужны `torch` и код из репозитория).